# ForecastLLM - Week 7 Day 3 and 4\n\n## Day 3 and Day 4: Training for Forecasting\n\nThis combined notebook preserves the original Day 3/4 training flow and adapts it from pricing to forecasting on one selected M4 hourly series.\n\nMulti-series training is deferred in this notebook.\n\nIf you are using `LITE_MODE=True`, use a smaller GPU profile. For full runs, use a larger GPU with enough VRAM.\n

In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")


In [9]:
import os
import re
from importlib import import_module
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
wandb = import_module("wandb")
wandb_login = getattr(wandb, "login")
wandb_init = getattr(wandb, "init")
wandb_finish = getattr(wandb, "finish")

from datasets import Dataset, DatasetDict
from dotenv import load_dotenv
from huggingface_hub import login
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

import week6.data_loader as data_loader


In [10]:
# Constants

load_dotenv()

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

PROJECT_NAME = "ForecastLLM"
RUN_NAME = "production"

LITE_MODE = os.getenv("LITE_MODE", "1").lower() in {"1", "true", "yes"}
BASE_MODEL = os.getenv("BASE_MODEL", "TinyLlama/TinyLlama-1.1B-Chat-v1.0")
DEFAULT_DATA_PATH = "/home/geo/Projects/Python/forecastllm/data/m4/processed/hourly_longest_series.csv"
DATA_PATH = os.getenv("FORECAST_DATA_PATH", DEFAULT_DATA_PATH)
SPLIT_DIR = os.getenv("FORECAST_SPLIT_DIR")
SEED = 42

FORECAST_HORIZON = 1
FEATURE_SET = ["lag_1", "lag_2", "lag_3", "lag_7", "lag_24", "day_of_week", "month"]

EPOCHS = 1 if LITE_MODE else 3
BATCH_SIZE = 4 if LITE_MODE else 16
GRADIENT_ACCUMULATION_STEPS = 2 if LITE_MODE else 1
MAX_SEQUENCE_LENGTH = 160
MAX_STEPS = 80 if LITE_MODE else 500

QUANT_4_BIT = torch.cuda.is_available()
LORA_R = 16 if LITE_MODE else 64
LORA_ALPHA = LORA_R * 2
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
LORA_DROPOUT = 0.1

LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = "cosine"
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

VAL_SIZE = 128 if LITE_MODE else 512
MIN_TOTAL_ROWS = 500 if LITE_MODE else 800
LOG_STEPS = 5 if LITE_MODE else 10
SAVE_STEPS = 25 if LITE_MODE else 50
EVAL_STEPS = SAVE_STEPS
LOG_TO_WANDB = True

ENABLE_HF_LOGIN = True
PUSH_TO_HUB = False

capability = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
use_bf16 = capability[0] >= 8

set_seed(SEED)

print(f"LITE_MODE={LITE_MODE} | BASE_MODEL={BASE_MODEL}")
print(f"FORECAST_DATA_PATH={DATA_PATH}")
print(f"FORECAST_SPLIT_DIR={SPLIT_DIR}")
print(f"MIN_TOTAL_ROWS={MIN_TOTAL_ROWS}")
print(f"Training plan: max_steps={MAX_STEPS}, batch_size={BATCH_SIZE}, grad_accum={GRADIENT_ACCUMULATION_STEPS}, eval_steps={EVAL_STEPS}")
print(f"CUDA available={torch.cuda.is_available()} | use_bf16={use_bf16}")

LITE_MODE=False | BASE_MODEL=TinyLlama/TinyLlama-1.1B-Chat-v1.0
FORECAST_DATA_PATH=/home/geo/Projects/Python/forecastllm/data/m4/processed/hourly_longest_series.csv
FORECAST_SPLIT_DIR=None
MIN_TOTAL_ROWS=800
Training plan: max_steps=500, batch_size=16, grad_accum=1, eval_steps=50
CUDA available=True | use_bf16=True


In [11]:
# A100-class GPUs support bf16; smaller GPUs may not
use_bf16


True

# More on Optimizers

https://huggingface.co/docs/transformers/main/en/perf_train_gpu_one#optimizers

We use AdamW-style optimizers for stable convergence while fine-tuning a forecasting assistant.


### Log in to Hugging Face and Weights & Biases

Dependencies are managed in `pyproject.toml` and installed with `uv`, not with notebook shell commands.
This notebook runs on Wintermute/PyCharm/Jupyter and reads secrets from `.env` using `python-dotenv` (not Colab secrets).

Required `.env` variables:
- `WANDB_API_KEY`
- `HF_TOKEN` or `HUGGINGFACE_TOKEN` (only if Hugging Face login/publish is enabled)


In [12]:
# Log in to HuggingFace (optional in this notebook)
if ENABLE_HF_LOGIN:
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("Hugging Face authentication enabled.")
    else:
        raise RuntimeError("Missing HF_TOKEN or HUGGINGFACE_TOKEN in .env")
else:
    print("Hugging Face login deferred in this notebook (training/evaluation runs locally).")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face authentication enabled.


Experiment tracking is important in forecasting because small metric shifts (MAE/sMAPE) can come from data split, feature set, or training changes.

In this notebook, Weights & Biases records run config and final forecasting metrics (`MAE`, `sMAPE`, model name, forecast horizon, feature set).

Hugging Face model publishing is deferred here (`PUSH_TO_HUB=False`) and left as a TODO so Day 3/4 stays focused on training/evaluation.

Evaluation remains one-step-ahead using observed lag features from each timestamp window; this is not full rolling-origin multistep forecasting yet.


In [13]:
# Log in to Weights & Biases
if LOG_TO_WANDB:
    if not WANDB_API_KEY:
        raise RuntimeError("Missing WANDB_API_KEY in .env")
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    wandb_login()

# Configure Weights & Biases for this run
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"


In [14]:
required_columns = {"timestamp", "value", *FEATURE_SET}

def _read_split_frame(split_dir: Path, split_name: str) -> pd.DataFrame:
    parquet_path = split_dir / f"{split_name}.parquet"
    csv_path = split_dir / f"{split_name}.csv"
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f"Missing split file for '{split_name}' in {split_dir}")

split_path = Path(SPLIT_DIR).expanduser().resolve() if SPLIT_DIR else None
if split_path and split_path.exists():
    train_df = _read_split_frame(split_path, "train")
    val_df = _read_split_frame(split_path, "val")
    test_df = _read_split_frame(split_path, "test")
    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        if not required_columns.issubset(split_df.columns):
            raise ValueError(
                f"Split '{split_name}' missing required columns. "
                f"Required={sorted(required_columns)}, got={sorted(split_df.columns)}"
            )
        split_df["timestamp"] = pd.to_datetime(split_df["timestamp"], errors="coerce")
        split_df["value"] = pd.to_numeric(split_df["value"], errors="coerce")
        split_df.dropna(subset=sorted(required_columns), inplace=True)
        split_df.sort_values("timestamp", inplace=True)
        split_df.reset_index(drop=True, inplace=True)

    if LITE_MODE:
        train_df = train_df.tail(min(len(train_df), 240)).reset_index(drop=True)

    supervised_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    ts_df = supervised_df[["timestamp", "value"]].copy().sort_values("timestamp").reset_index(drop=True)
    print(
        f"Loaded split artifacts from {split_path} | "
        f"train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}"
    )
else:
    data_loader = import_module("week6.data_loader")
    data_loader = import_module("importlib").reload(data_loader)
    loaded = data_loader.load_sample_series()
    if not isinstance(loaded, pd.DataFrame):
        raise TypeError(f"load_sample_series must return a pandas DataFrame, got {type(loaded)}")
    if not {"timestamp", "value"}.issubset(loaded.columns):
        raise ValueError("Loaded dataframe must include 'timestamp' and 'value' columns")

    ts_df = loaded[["timestamp", "value"]].copy()
    ts_df["timestamp"] = pd.to_datetime(ts_df["timestamp"], errors="coerce")
    ts_df["value"] = pd.to_numeric(ts_df["value"], errors="coerce")
    ts_df = ts_df.dropna(subset=["timestamp", "value"]).sort_values("timestamp").reset_index(drop=True)

    supervised_df = ts_df.copy()
    supervised_df["lag_1"] = supervised_df["value"].shift(1)
    supervised_df["lag_2"] = supervised_df["value"].shift(2)
    supervised_df["lag_3"] = supervised_df["value"].shift(3)
    supervised_df["lag_7"] = supervised_df["value"].shift(7)
    supervised_df["lag_24"] = supervised_df["value"].shift(24)
    supervised_df["day_of_week"] = supervised_df["timestamp"].dt.dayofweek
    supervised_df["month"] = supervised_df["timestamp"].dt.month
    supervised_df = supervised_df.dropna().reset_index(drop=True)

    if LITE_MODE:
        supervised_df = supervised_df.tail(min(len(supervised_df), 240)).reset_index(drop=True)

    n = len(supervised_df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    train_df = supervised_df.iloc[:train_end].copy()
    val_df = supervised_df.iloc[train_end:val_end].copy()
    test_df = supervised_df.iloc[val_end:].copy()
    print(f"Loaded raw dataset source={DATA_PATH or os.getenv('FORECAST_DATA_PATH')} | rows={len(ts_df)}")

if len(supervised_df) < MIN_TOTAL_ROWS:
    raise RuntimeError(
        f"Insufficient rows for substantial training: {len(supervised_df)} < {MIN_TOTAL_ROWS}. "
        "Provide larger raw data via FORECAST_DATA_PATH/DATA_PATH or split artifacts via FORECAST_SPLIT_DIR."
    )

def build_prompt_from_row(feature_row):
    return (
        "You are a forecasting assistant.\n"
        "Given lag and calendar context, predict the next hourly value.\n"
        f"lag_1={feature_row['lag_1']:.3f}, lag_2={feature_row['lag_2']:.3f}, lag_3={feature_row['lag_3']:.3f}, "
        f"lag_7={feature_row['lag_7']:.3f}, lag_24={feature_row['lag_24']:.3f}, "
        f"day_of_week={int(feature_row['day_of_week'])}, month={int(feature_row['month'])}.\n"
        "Return only a numeric forecast."
    )

def build_completion_from_row(feature_row):
    return f"{feature_row['value']:.3f}"

def make_records(df, include_completion):
    records = []
    for _, record_row in df.iterrows():
        record_prompt_text = build_prompt_from_row(record_row)
        record_completion_text = build_completion_from_row(record_row)
        rec = {"prompt": record_prompt_text, "completion": record_completion_text}
        rec["text"] = f"{record_prompt_text}\n{record_completion_text}" if include_completion else record_prompt_text
        records.append(rec)
    return records

train_records = make_records(train_df, include_completion=True)
val_records = make_records(val_df, include_completion=True)
test_records = make_records(test_df, include_completion=False)

if VAL_SIZE < len(val_records):
    val_records = val_records[:VAL_SIZE]

dataset = DatasetDict({
    "train": Dataset.from_pandas(pd.DataFrame(train_records), preserve_index=False),
    "val": Dataset.from_pandas(pd.DataFrame(val_records), preserve_index=False),
    "test": Dataset.from_pandas(pd.DataFrame(test_records), preserve_index=False),
})

print(f"Series rows={len(ts_df):,} | supervised rows={len(supervised_df):,}")
print(f"train={len(train_records):,}, val={len(val_records):,}, test={len(test_records):,}")
dataset

Loaded raw dataset source=/home/geo/Projects/Python/forecastllm/data/m4/processed/hourly_longest_series.csv | rows=960
Series rows=960 | supervised rows=936
train=655, val=140, test=141


DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion', 'text'],
        num_rows: 655
    })
    val: Dataset({
        features: ['prompt', 'completion', 'text'],
        num_rows: 140
    })
    test: Dataset({
        features: ['prompt', 'completion', 'text'],
        num_rows: 141
    })
})

In [ ]:
# if you wish to reduce the training dataset further, uncomment:
# dataset['train'] = dataset['train'].select(range(min(1000, len(dataset['train']))))


In [15]:
wandb_run = None
if LOG_TO_WANDB:
    current_run = getattr(wandb, "run", None)
    if current_run is None:
        wandb_run = wandb_init(
            project="ForecastLLM",
            name="week7-day3-and-4",
            config={
                "base_model": BASE_MODEL,
                "lite_mode": LITE_MODE,
                "forecast_horizon": FORECAST_HORIZON,
                "feature_set": FEATURE_SET,
                "train_rows": len(dataset['train']),
                "val_rows": len(dataset['val']),
                "test_rows": len(dataset['test']),
            },
        )
    else:
        wandb_run = current_run
        print(f"Reusing active W&B run: {wandb_run.id}")


## Now load the Tokenizer and Model

We use quantization when CUDA is available to keep memory use lower during QLoRA-style fine-tuning.


In [16]:
# pick the right quantization
if QUANT_4_BIT:
    quant_kwargs: dict[str, object] = {
        "load_in_4bit": True,
        "bnb_4bit_use_double_quant": True,
        "bnb_4bit_compute_dtype": torch.bfloat16 if use_bf16 else torch.float16,
        "bnb_4bit_quant_type": "nf4",
    }
    quant_config = BitsAndBytesConfig(**quant_kwargs)
else:
    quant_config = None

print("Using 4-bit quantization:", bool(quant_config is not None))


Using 4-bit quantization: True


In [17]:
# Load the Tokenizer and the Model
from_pretrained = getattr(AutoTokenizer, "from_pretrained", None)
if from_pretrained is None:
    raise RuntimeError("AutoTokenizer.from_pretrained is unavailable")

tokenizer = from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer is None:
    raise RuntimeError(f"Tokenizer load failed for {BASE_MODEL}")

eos_token = getattr(tokenizer, "eos_token", None)
if tokenizer.pad_token is None:
    if eos_token is None:
        raise RuntimeError("Tokenizer is missing both pad_token and eos_token")
    tokenizer.pad_token = eos_token
tokenizer.padding_side = "right"

infer_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (torch.float16 if torch.cuda.is_available() else torch.float32)

model_kwargs: dict[str, object] = {"trust_remote_code": True}
if HF_TOKEN:
    model_kwargs["token"] = HF_TOKEN

if quant_config is not None:
    model_kwargs["quantization_config"] = quant_config
    model_kwargs["torch_dtype"] = infer_dtype
    model_kwargs["device_map"] = "auto"
else:
    model_kwargs["torch_dtype"] = infer_dtype

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
pad_token_id = getattr(tokenizer, "pad_token_id", None)
if pad_token_id is None:
    raise RuntimeError("Tokenizer pad_token_id is missing")
if getattr(base_model, "config", None) is not None:
    base_model.config.pad_token_id = pad_token_id
if getattr(base_model, "generation_config", None) is not None:
    base_model.generation_config.pad_token_id = pad_token_id

if hasattr(base_model, "get_memory_footprint"):
    print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/home/geo/Projects/Python/forecastllm/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Memory footprint: 746.8 MB


# AND NOW

## We set up the configuration for Training

We create:

- a `LoraConfig` with LoRA hyperparameters
- an `SFTConfig` with training parameters


In [18]:
# LoRA Parameters
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
lora_parameters


LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=64, target_modules={'o_proj', 'q_proj', 'k_proj', 'v_proj'}, exclude_modules=None, lora_alpha=128, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)

In [19]:
# Training parameters
output_dir = f"forecastllm-{datetime.now():%Y-%m-%d_%H.%M.%S}"

train_parameters = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=not use_bf16,
    bf16=use_bf16,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.3,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else "none",
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    save_strategy="steps",
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    push_to_hub=PUSH_TO_HUB,
)

train_parameters


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


SFTConfig(output_dir='forecastllm-2026-05-01_11.11.05', per_device_train_batch_size=16, num_train_epochs=3, max_steps=500, learning_rate=0.0001, lr_scheduler_type=<SchedulerType.COSINE: 'cosine'>, lr_scheduler_kwargs=None, warmup_steps=0.01, optim=<OptimizerNames.PAGED_ADAMW: 'paged_adamw_32bit'>, optim_args=None, weight_decay=0.001, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=1, average_tokens_across_devices=True, max_grad_norm=0.3, label_smoothing_factor=0.0, bf16=True, fp16=False, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False}, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=None, auto_find_batch_size=False, logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_steps=10, logging_fir

# AND NOW - create the trainer


In [20]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    peft_config=lora_parameters,
    args=train_parameters,
)
fine_tuning


Adding EOS to train dataset:   0%|          | 0/655 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/655 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/655 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/140 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/140 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/140 [00:00<?, ? examples/s]

## In the next cell, we kick off fine-tuning and evaluate forecasts

The forecast evaluation below is one-step-ahead over the test windows (not full rolling-origin multistep).


In [21]:
# Fine-tune, then evaluate forecast quality
if LOG_TO_WANDB and getattr(wandb, "run", None) is None:
    wandb_run = wandb_init(
        project="ForecastLLM",
        name="week7-day3-and-4",
        config={
            "base_model": BASE_MODEL,
            "lite_mode": LITE_MODE,
            "forecast_horizon": FORECAST_HORIZON,
            "feature_set": FEATURE_SET,
            "train_rows": len(dataset['train']),
            "val_rows": len(dataset['val']),
            "test_rows": len(dataset['test']),
        },
    )

train_result = fine_tuning.train()

def extract_first_number(text):
    m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", text)
    return float(m.group(0)) if m else np.nan

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom == 0, 1e-9, denom)
    return 100 * np.mean(2 * np.abs(y_true - y_pred) / denom)

eval_rows = min(len(test_df), 32 if LITE_MODE else 128)
eval_slice = test_df.iloc[:eval_rows].copy()

predictions = []
actuals = []

pad_token_id = getattr(tokenizer, "pad_token_id", None)
if pad_token_id is None:
    raise RuntimeError("Tokenizer pad_token_id is missing")

trained_model = fine_tuning.model
if trained_model is None:
    raise RuntimeError("Trainer model is unavailable after fine-tuning")

if getattr(trained_model, "config", None) is not None:
    trained_model.config.pad_token_id = pad_token_id
if getattr(trained_model, "generation_config", None) is not None:
    trained_model.generation_config.pad_token_id = pad_token_id

# Keep projection/input embedding dtype aligned with runtime compute dtype for generation
base_model = trained_model.get_base_model() if hasattr(trained_model, "get_base_model") else trained_model
model_device = next(trained_model.parameters()).device
runtime_dtype = torch.bfloat16 if (model_device.type == "cuda" and torch.cuda.is_bf16_supported()) else (torch.float16 if model_device.type == "cuda" else torch.float32)

modules_to_align = []
if hasattr(trained_model, "get_output_embeddings"):
    modules_to_align.append(trained_model.get_output_embeddings())
if hasattr(base_model, "get_output_embeddings"):
    modules_to_align.append(base_model.get_output_embeddings())
if hasattr(trained_model, "get_input_embeddings"):
    modules_to_align.append(trained_model.get_input_embeddings())
if hasattr(base_model, "get_input_embeddings"):
    modules_to_align.append(base_model.get_input_embeddings())

for module in modules_to_align:
    if module is not None:
        module.to(device=model_device, dtype=runtime_dtype)

# Llama path used in the traceback (transformers.models.llama.modeling_llama)
for attr_path in [
    "lm_head",
    "model.lm_head",
    "base_model.lm_head",
    "base_model.model.lm_head",
]:
    obj = trained_model
    ok = True
    for part in attr_path.split("."):
        if not hasattr(obj, part):
            ok = False
            break
        obj = getattr(obj, part)
    if ok and obj is not None:
        obj.to(device=model_device, dtype=runtime_dtype)

lm_head = base_model.get_output_embeddings() if hasattr(base_model, "get_output_embeddings") else None
if lm_head is not None and hasattr(lm_head, "weight"):
    print(f"Generation dtype check -> lm_head: {lm_head.weight.dtype}, target: {runtime_dtype}")
trained_model.eval()
if getattr(trained_model, "generation_config", None) is not None:
    trained_model.generation_config.max_length = None

generate_kwargs = {
    "max_new_tokens": 16,
    "do_sample": False,
    "pad_token_id": pad_token_id,
    "max_length": None,
}

for _, eval_row in eval_slice.iterrows():
    eval_prompt_text = build_prompt_from_row(eval_row)
    inputs = tokenizer(eval_prompt_text, return_tensors="pt")
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    with torch.no_grad():
        if model_device.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=runtime_dtype):
                out = trained_model.generate(
                    **inputs,
                    **generate_kwargs,
                )
        else:
            out = trained_model.generate(
                **inputs,
                **generate_kwargs,
            )

    generated_ids = out[0][inputs['input_ids'].shape[1]:]
    completion = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    pred = extract_first_number(completion)

    predictions.append(pred)
    actuals.append(float(eval_row['value']))

eval_df = pd.DataFrame({"actual": actuals, "pred": predictions}).dropna()
if eval_df.empty:
    raise RuntimeError("No numeric forecasts could be parsed from model outputs.")

mae = float(np.mean(np.abs(eval_df['actual'] - eval_df['pred'])))
smape_value = float(smape(eval_df['actual'], eval_df['pred']))

baseline_eval_df = eval_slice[["value", "lag_1"]].rename(columns={"value": "actual", "lag_1": "pred"}).copy()
baseline_eval_df = baseline_eval_df.dropna()
baseline_mae = float(np.mean(np.abs(baseline_eval_df['actual'] - baseline_eval_df['pred'])))
baseline_smape = float(smape(baseline_eval_df['actual'], baseline_eval_df['pred']))

metrics = {
    "MAE": mae,
    "sMAPE": smape_value,
    "baseline_MAE": baseline_mae,
    "baseline_sMAPE": baseline_smape,
    "delta_MAE_vs_naive": baseline_mae - mae,
    "delta_sMAPE_vs_naive": baseline_smape - smape_value,
    "model_name": BASE_MODEL,
    "forecast_horizon": FORECAST_HORIZON,
    "feature_set": ",".join(FEATURE_SET),
    "eval_rows": int(len(eval_df)),
}

print(metrics)

if LOG_TO_WANDB:
    if getattr(wandb, "run", None) is not None:
        wandb.log(metrics)
    else:
        print("Skipping wandb.log because no active W&B run was found.")

if PUSH_TO_HUB:
    if HF_TOKEN:
        trained_model.push_to_hub(output_dir, private=True)
        print(f"Saved to the hub: {output_dir}")
    else:
        raise RuntimeError("Missing HF_TOKEN or HUGGINGFACE_TOKEN in .env")
else:
    print("TODO: Hugging Face model publishing deferred for this Day 3/4 forecasting run.")


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.236815,0.290669,1.480396,85493.000000,0.890476
100,0.204261,0.331266,1.562405,170986.000000,0.880952
150,0.194548,0.370707,1.416692,256479.000000,0.880952
200,0.168056,0.428492,1.390124,341972.000000,0.872619
250,0.128058,0.491155,1.304757,427358.000000,0.883333
300,0.066904,0.808369,1.215162,512851.000000,0.878571
350,0.048917,0.889834,1.136265,598344.000000,0.880952
400,0.022023,1.002380,1.062241,683837.000000,0.875000
450,0.011570,1.097161,1.026841,769330.000000,0.879762
500,0.006777,1.108256,1.024127,854716.000000,0.879762


Generation dtype check -> lm_head: torch.bfloat16, target: torch.bfloat16


/home/geo/Projects/Python/forecastllm/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'MAE': 0.09687499999999999, 'sMAPE': 0.45436108597806363, 'baseline_MAE': 0.7640625000000001, 'baseline_sMAPE': 3.6425828417173025, 'delta_MAE_vs_naive': 0.6671875, 'delta_sMAPE_vs_naive': 3.1882217557392387, 'model_name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'forecast_horizon': 1, 'feature_set': 'lag_1,lag_2,lag_3,lag_7,lag_24,day_of_week,month', 'eval_rows': 128}
TODO: Hugging Face model publishing deferred for this Day 3/4 forecasting run.


In [22]:
if LOG_TO_WANDB and wandb_run is not None:
    wandb_finish()


MAE,▁
baseline_MAE,▁
baseline_sMAPE,▁
delta_MAE_vs_naive,▁
delta_sMAPE_vs_naive,▁
eval/entropy,▇█▆▆▅▃▂▁▁▁
eval/loss,▁▁▂▂▃▅▆▇██
eval/mean_token_accuracy,█▄▄▁▅▃▄▂▄▄
eval/num_tokens,▁▂▃▃▄▅▆▆▇█
eval/runtime,▃▇▆▅▂▅▁▄▂█
+13,...
